<div style="text-align:center; padding:20px 0">
<img src="https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/media/logo_dataprojectlab.png" width="220"/>
</div>

# HotelChain West Analytics
## Notebook 1 — SQL Analytics : KPIs, Performance & Analyses Avancées

> **Prérequis** : avoir lu le contexte métier et le dictionnaire des données sur la plateforme.

| | |
|---|---|
| **Niveau** | Avancé |
| **Outils** | Python, DuckDB (JupySQL), pandas |
| **Durée estimée** | 4h à 5h |

---
## 0. Mise en place de l'environnement

In [ ]:
!pip install jupysql==0.11.1 duckdb-engine --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import duckdb
import os, sys, warnings

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", "{:.2f}".format)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor":   "#F9F9F8",
    "axes.grid":        True,
    "grid.alpha":       0.35,
    "font.size":        11,
})

COLORS = {
    "primary":   "#534AB7",
    "secondary": "#1D9E75",
    "warning":   "#EF9F27",
    "danger":    "#E24B4A",
    "neutral":   "#888780",
}

# ── Détection Colab / Local ──────────────────────────────────────────────────
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    SAVE_PATH = "/content/drive/MyDrive/DataProjectLab/projects/hotelchain_analytics/"
else:
    SAVE_PATH = "./outputs/"
os.makedirs(SAVE_PATH, exist_ok=True)
print(f"📁 Environnement : {'Colab' if IN_COLAB else 'Local'}")
print(f"📁 Dossier       : {SAVE_PATH}")
print("Environnement prêt ✅")

---
## 1. Connexion DuckDB et chargement des tables

> **MÉTHODE — DuckDB remplace SQL Server dans ce notebook.**
>
> DuckDB est une base de données SQL embarquée optimisée pour l'analytique. Elle charge les CSV directement depuis GitHub via `read_csv_auto()` — zéro installation, compatible Colab et local.
>
> **Adaptations syntaxiques vs SQL Server :**
>
> | SQL Server | DuckDB |
> |---|---|
> | `CREATE OR ALTER VIEW` | `CREATE OR REPLACE VIEW` |
> | `TOP N` | `LIMIT N` |
> | `EOMONTH()` | `LAST_DAY()` |
> | `PIVOT` | pandas `.pivot_table()` |
> | `CREATE PROCEDURE` | fonction Python |
> | `BULK INSERT` | `read_csv_auto()` |

In [ ]:
BASE_URL = "https://raw.githubusercontent.com/dataprojectlabs/DataProjectLab-projects/refs/heads/main/projets/hotelchain_analytics/data/"

# Créer la connexion et charger les 6 tables
conn = duckdb.connect()
conn.execute(f"""
    CREATE TABLE hotels       AS SELECT * FROM read_csv_auto('{BASE_URL}hotels.csv');
    CREATE TABLE chambres     AS SELECT * FROM read_csv_auto('{BASE_URL}chambres.csv');
    CREATE TABLE clients      AS SELECT * FROM read_csv_auto('{BASE_URL}clients.csv');
    CREATE TABLE reservations AS SELECT * FROM read_csv_auto('{BASE_URL}reservations.csv', timestampformat='%Y-%m-%d');
    CREATE TABLE paiements    AS SELECT * FROM read_csv_auto('{BASE_URL}paiements.csv');
    CREATE TABLE services     AS SELECT * FROM read_csv_auto('{BASE_URL}services.csv');
""")

# Vérification
result = conn.execute("""
    SELECT 'hotels'       AS table_name, COUNT(*) AS nb_lignes FROM hotels       UNION ALL
    SELECT 'chambres',      COUNT(*) FROM chambres     UNION ALL
    SELECT 'clients',       COUNT(*) FROM clients      UNION ALL
    SELECT 'reservations',  COUNT(*) FROM reservations UNION ALL
    SELECT 'paiements',     COUNT(*) FROM paiements    UNION ALL
    SELECT 'services',      COUNT(*) FROM services
""").df()
display(result)
print("✅ 6 tables chargées")

In [ ]:
# Lier DuckDB à JupySQL pour les cellules %%sql
%load_ext sql
%sql conn --alias duckdb
%config SqlMagic.autopandas = True
%config SqlMagic.feedback = False
print("%%sql prêt ✅")

---
## 2. Vues de nettoyage

> **MÉTHODE — Pourquoi des vues plutôt que modifier les données brutes ?**
>
> Les vues sont des requêtes sauvegardées qui se comportent comme des tables. Les données originales restent intactes et reproductibles. Toute l'analyse s'appuie sur ces vues nettoyées.

### Vue 1 — Clients propres (sans doublons email, âges valides)

In [ ]:
%%sql
CREATE OR REPLACE VIEW vw_clients_propres AS
SELECT *
FROM clients
WHERE age BETWEEN 16 AND 80
  AND client_id NOT IN (
      -- Exclure les doublons email : garder le plus petit client_id
      SELECT client_id
      FROM (
          SELECT client_id,
                 ROW_NUMBER() OVER (PARTITION BY email ORDER BY client_id) AS rn
          FROM clients
      ) sub
      WHERE rn > 1
  );

### Vue 2 — Réservations propres (montants > 0, nb_nuits > 0)

In [ ]:
%%sql
CREATE OR REPLACE VIEW vw_reservations_propres AS
SELECT *
FROM reservations
WHERE nb_nuits > 0
  AND montant_total > 0;

### Vue 3 — Paiements valides

In [ ]:
%%sql
CREATE OR REPLACE VIEW vw_paiements_valides AS
SELECT *
FROM paiements
WHERE statut_paiement = 'Valide';

In [ ]:
# Vérification avant/après nettoyage
check = conn.execute("""
    SELECT 'clients brut'        AS source, COUNT(*) AS nb FROM clients         UNION ALL
    SELECT 'clients propres',      COUNT(*) FROM vw_clients_propres             UNION ALL
    SELECT 'reservations brut',    COUNT(*) FROM reservations                   UNION ALL
    SELECT 'reservations propres', COUNT(*) FROM vw_reservations_propres        UNION ALL
    SELECT 'paiements brut',       COUNT(*) FROM paiements                      UNION ALL
    SELECT 'paiements valides',    COUNT(*) FROM vw_paiements_valides
""").df()
display(check)

---
## 3. KPIs globaux en une seule requête

> **MÉTHODE — Une seule requête pour tous les KPIs de surface.**
>
> C'est la première réponse au Directeur Général : *'Quel est l'état de mon business ?'*  
> On filtre sur les réservations avec statut `'Terminee'` pour les KPIs financiers.

In [ ]:
%%sql df_kpi <<
SELECT
    COUNT(DISTINCT r.reservation_id)                                           AS total_reservations,
    ROUND(SUM(p.montant), 0)                                                   AS revenu_total,
    ROUND(AVG(CAST(r.csat AS DOUBLE)), 2)                                      AS csat_moyen,
    ROUND(AVG(CAST(r.nb_nuits AS DOUBLE)), 2)                                  AS duree_moyenne_sejour,
    COUNT(DISTINCT r.client_id)                                                AS nb_clients_uniques,
    ROUND(SUM(p.montant) / COUNT(DISTINCT r.reservation_id), 0)               AS revenu_moyen_par_reservation,
    ROUND(
        SUM(CASE WHEN r.statut = 'Annulee' THEN 1.0 ELSE 0.0 END)
        / COUNT(*) * 100, 1
    )                                                                          AS taux_annulation_pct
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id

---
## 4. Taux d'occupation par hôtel et par mois

> **MÉTHODE — Taux d'occupation = nuits vendues / (nb_chambres × nb_jours_du_mois) × 100**
>
> On utilise `RANK() OVER (PARTITION BY annee, mois ORDER BY taux DESC)` pour classer les hôtels chaque mois — sans perdre les lignes des autres hôtels, contrairement à `LIMIT`.
>
> **DuckDB :** `LAST_DAY(DATE_TRUNC('month', date_arrivee))` remplace `EOMONTH()` de SQL Server.

In [ ]:
%%sql df_occupation <<
WITH nuits_vendues AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        SUM(r.nb_nuits)       AS nuits_vendues
    FROM vw_reservations_propres r
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
capacite AS (
    -- Capacité théorique = nb_chambres × nb_jours_du_mois
    SELECT
        nv.hotel_id,
        nv.annee,
        nv.mois,
        h.nb_chambres,
        DAY(LAST_DAY(MAKE_DATE(nv.annee, nv.mois, 1))) AS nb_jours_mois,
        h.nb_chambres * DAY(LAST_DAY(MAKE_DATE(nv.annee, nv.mois, 1))) AS capacite_theorique
    FROM nuits_vendues nv
    JOIN hotels h ON nv.hotel_id = h.hotel_id
)
SELECT
    h.nom,
    nv.annee,
    nv.mois,
    nv.nuits_vendues,
    c.capacite_theorique,
    ROUND(nv.nuits_vendues * 100.0 / c.capacite_theorique, 1) AS taux_occupation_pct,
    RANK() OVER (
        PARTITION BY nv.annee, nv.mois
        ORDER BY nv.nuits_vendues * 100.0 / c.capacite_theorique DESC
    ) AS rang_mois
FROM nuits_vendues nv
JOIN hotels h    ON nv.hotel_id = h.hotel_id
JOIN capacite c  ON nv.hotel_id = c.hotel_id AND nv.annee = c.annee AND nv.mois = c.mois
ORDER BY nv.annee, nv.mois, taux_occupation_pct DESC

In [ ]:
# Visualisation : taux d'occupation mensuel par hôtel
df_occ_pivot = df_occupation.pivot_table(
    index=['annee', 'mois'], columns='nom', values='taux_occupation_pct'
).reset_index()
df_occ_pivot['periode'] = df_occ_pivot['annee'].astype(str) + '-' + df_occ_pivot['mois'].astype(str).str.zfill(2)

hotels_cols = [c for c in df_occ_pivot.columns if c not in ['annee', 'mois', 'periode']]
fig, ax = plt.subplots(figsize=(14, 5))
for col in hotels_cols:
    ax.plot(df_occ_pivot['periode'], df_occ_pivot[col], marker='o', markersize=3, linewidth=1.5, label=col)
ax.axhline(70, color=COLORS['secondary'], linestyle='--', linewidth=1.5, label='Cible 70%')
ax.axhline(40, color=COLORS['danger'], linestyle='--', linewidth=1.5, label='Seuil alerte 40%')
ax.set_xticks(range(0, len(df_occ_pivot), 3))
ax.set_xticklabels(df_occ_pivot['periode'].iloc[::3], rotation=45, ha='right', fontsize=9)
ax.set_title('Taux d\'occupation mensuel par hôtel', fontweight='bold', fontsize=13)
ax.set_ylabel('Taux d\'occupation (%)')
ax.legend(fontsize=9)
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}taux_occupation.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 5. Performance par canal — RANK() et LAG()

> **MÉTHODE — Deux questions distinctes sur les canaux :**
>
> **RANK() :** quel canal est le plus rentable globalement ?  
> **LAG() :** comment évolue chaque canal mois par mois ?

In [ ]:
%%sql df_canal <<
SELECT
    r.canal,
    COUNT(*)                                                                   AS nb_reservations,
    ROUND(SUM(p.montant), 0)                                                   AS revenu_total,
    ROUND(AVG(p.montant), 0)                                                   AS revenu_moyen,
    ROUND(
        SUM(CASE WHEN r.statut = 'Annulee' THEN 1.0 ELSE 0.0 END)
        / COUNT(*) * 100, 1
    )                                                                          AS taux_annulation_pct,
    RANK() OVER (ORDER BY SUM(p.montant) DESC)                                 AS rang_revenu
FROM vw_reservations_propres r
LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
GROUP BY r.canal
ORDER BY revenu_total DESC

In [ ]:
%%sql df_canal_lag <<
-- Tendance mensuelle par canal avec LAG()
WITH mensuel AS (
    SELECT
        r.canal,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        ROUND(SUM(p.montant), 0) AS revenu_mensuel
    FROM vw_reservations_propres r
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    GROUP BY r.canal, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
)
SELECT
    canal,
    annee,
    mois,
    revenu_mensuel,
    LAG(revenu_mensuel) OVER (PARTITION BY canal ORDER BY annee, mois) AS revenu_mois_prec,
    ROUND(
        revenu_mensuel - LAG(revenu_mensuel) OVER (PARTITION BY canal ORDER BY annee, mois), 0
    ) AS variation
FROM mensuel
ORDER BY canal, annee, mois

---
## 6. RevPAR — Revenue Per Available Room

> **MÉTHODE — RevPAR = ADR × Taux d'occupation**
>
> C'est l'indicateur standard de l'industrie hôtelière. Il combine à la fois la politique tarifaire (ADR) et le remplissage (taux d'occupation) en une seule métrique.
>
> - **ADR élevé + taux bas** : hotel trop cher, peu rempli
> - **ADR bas + taux haut** : hôtel sous-tarifé, bien rempli mais peu rentable
> - **ADR élevé + taux élevé** : sweet spot tarifaire

In [ ]:
%%sql df_revpar <<
WITH adr_calc AS (
    SELECT
        r.hotel_id,
        YEAR(r.date_arrivee)  AS annee,
        MONTH(r.date_arrivee) AS mois,
        ROUND(SUM(p.montant) / NULLIF(SUM(r.nb_nuits), 0), 0) AS adr,
        SUM(r.nb_nuits)                                        AS nuits_vendues
    FROM vw_reservations_propres r
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY r.hotel_id, YEAR(r.date_arrivee), MONTH(r.date_arrivee)
),
capacite AS (
    SELECT
        a.hotel_id, a.annee, a.mois,
        h.nb_chambres * DAY(LAST_DAY(MAKE_DATE(a.annee, a.mois, 1))) AS capacite_theorique
    FROM adr_calc a
    JOIN hotels h ON a.hotel_id = h.hotel_id
)
SELECT
    h.nom,
    a.annee,
    a.mois,
    a.adr,
    ROUND(a.nuits_vendues * 100.0 / c.capacite_theorique, 1) AS taux_occupation_pct,
    ROUND(a.adr * (a.nuits_vendues * 1.0 / c.capacite_theorique), 0) AS revpar,
    RANK() OVER (
        PARTITION BY a.annee, a.mois
        ORDER BY a.adr * (a.nuits_vendues * 1.0 / c.capacite_theorique) DESC
    ) AS rang_revpar
FROM adr_calc a
JOIN hotels h    ON a.hotel_id = h.hotel_id
JOIN capacite c  ON a.hotel_id = c.hotel_id AND a.annee = c.annee AND a.mois = c.mois
ORDER BY a.annee, a.mois, revpar DESC

---
## 7. Segmentation clients — NTILE et valeur client (CLV)

> **MÉTHODE — `NTILE(4) OVER (ORDER BY revenu_total DESC)` : segmentation en quartiles.**
>
> Quartile 1 = top 25% des clients en valeur → segment à protéger.  
> `PERCENT_RANK()` pour identifier le top 10% (VIP).

In [ ]:
%%sql df_clv <<
WITH clv AS (
    SELECT
        c.client_id,
        c.prenom || ' ' || c.nom     AS client_nom,
        c.nationalite,
        c.client_fidele,
        COUNT(r.reservation_id)      AS nb_sejours,
        ROUND(SUM(p.montant), 0)     AS revenu_total,
        ROUND(AVG(CAST(r.nb_nuits AS DOUBLE)), 1) AS duree_moyenne,
        ROUND(AVG(CAST(r.csat AS DOUBLE)), 2)     AS csat_moyen
    FROM vw_clients_propres c
    JOIN vw_reservations_propres r   ON c.client_id = r.client_id
    LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
    WHERE r.statut IN ('Terminee', 'En cours')
    GROUP BY c.client_id, c.prenom, c.nom, c.nationalite, c.client_fidele
)
SELECT *,
    NTILE(4) OVER (ORDER BY revenu_total DESC)    AS quartile_valeur,
    CASE
        WHEN PERCENT_RANK() OVER (ORDER BY revenu_total DESC) <= 0.10
        THEN 'VIP' ELSE 'Standard'
    END AS segment_vip
FROM clv
ORDER BY revenu_total DESC
LIMIT 20

---
## 8. Heatmap saisonnalité — mois × hôtel

> **MÉTHODE — Le PIVOT SQL Server n'existe pas dans DuckDB.**
>
> On utilise `pandas.pivot_table()` — résultat identique, code plus lisible.

In [ ]:
# Récupérer les données pour la heatmap
df_heatmap_raw = conn.execute("""
    SELECT
        h.nom                                                          AS hotel,
        MONTH(r.date_arrivee)                                          AS mois,
        ROUND(
            SUM(r.nb_nuits) * 100.0 /
            (h.nb_chambres * DAY(LAST_DAY(MAKE_DATE(YEAR(r.date_arrivee), MONTH(r.date_arrivee), 1)))),
        1) AS taux_occupation
    FROM vw_reservations_propres r
    JOIN hotels h ON r.hotel_id = h.hotel_id
    WHERE r.statut IN ('Terminee', 'En cours')
      AND YEAR(r.date_arrivee) = 2023
    GROUP BY h.nom, h.nb_chambres, MONTH(r.date_arrivee), YEAR(r.date_arrivee)
""").df()

In [ ]:
import seaborn as sns

# Pivot pandas
pivot = df_heatmap_raw.pivot_table(index='hotel', columns='mois', values='taux_occupation')
pivot.columns = ['Jan','Fév','Mar','Avr','Mai','Jun','Jul','Aoû','Sep','Oct','Nov','Déc']

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(
    pivot, annot=True, fmt='.0f', cmap='RdYlGn',
    vmin=30, vmax=90, linewidths=0.5, ax=ax,
    cbar_kws={'label': 'Taux occupation (%)'}
)
ax.set_title('Heatmap saisonnalité — Taux d\'occupation 2023 (mois × hôtel)', fontweight='bold', fontsize=13)
ax.set_xlabel('Mois')
ax.set_ylabel('Hôtel')
plt.tight_layout()
plt.savefig(f"{SAVE_PATH}heatmap_saisonnalite.png", dpi=150, bbox_inches='tight')
plt.show()

---
## 9. Rapport hôtel — fonction Python (remplace la procédure stockée)

> **MÉTHODE — `CREATE PROCEDURE` n'existe pas dans DuckDB.**
>
> On crée une fonction Python qui accepte un `hotel_id` et retourne le rapport complet — exactement le même résultat qu'une procédure stockée SQL Server, avec plus de flexibilité.

In [ ]:
def rapport_hotel(hotel_id: str, annee: int = 2023):
    """Génère le rapport complet pour un hôtel — équivalent de sp_rapport_hotel."""
    return conn.execute(f"""
        WITH base AS (
            SELECT
                h.nom,
                MONTH(r.date_arrivee)                                          AS mois,
                COUNT(DISTINCT r.reservation_id)                               AS nb_reservations,
                SUM(r.nb_nuits)                                                AS nuits_vendues,
                h.nb_chambres * DAY(LAST_DAY(MAKE_DATE(
                    YEAR(r.date_arrivee), MONTH(r.date_arrivee), 1)))          AS capacite,
                ROUND(SUM(p.montant), 0)                                       AS revenu,
                ROUND(AVG(CAST(r.csat AS DOUBLE)), 2)                          AS csat_moyen,
                ROUND(SUM(p.montant) / NULLIF(SUM(r.nb_nuits), 0), 0)         AS adr
            FROM vw_reservations_propres r
            JOIN hotels h    ON r.hotel_id = h.hotel_id
            LEFT JOIN vw_paiements_valides p ON r.reservation_id = p.reservation_id
            WHERE r.hotel_id = '{hotel_id}'
              AND YEAR(r.date_arrivee) = {annee}
              AND r.statut IN ('Terminee', 'En cours')
            GROUP BY h.nom, h.nb_chambres, MONTH(r.date_arrivee), YEAR(r.date_arrivee)
        )
        SELECT *,
            ROUND(nuits_vendues * 100.0 / capacite, 1) AS taux_occupation_pct,
            ROUND(adr * nuits_vendues * 1.0 / capacite, 0) AS revpar
        FROM base
        ORDER BY mois
    """).df()

# Test sur le premier hôtel
df_rapport = rapport_hotel('HTL001', 2023)
display(df_rapport)

---
## 10. Export des datasets analytiques

> Ces fichiers seront importés dans Power BI au Notebook 2.

In [ ]:
# Export des datasets principaux
df_occupation.to_csv(f"{SAVE_PATH}hotelchain_occupation.csv", index=False)
df_revpar.to_csv(f"{SAVE_PATH}hotelchain_revpar.csv", index=False)
df_clv.to_csv(f"{SAVE_PATH}hotelchain_clients_clv.csv", index=False)
df_canal.to_csv(f"{SAVE_PATH}hotelchain_canaux.csv", index=False)

print("Fichiers exportés ✅")
print("  hotelchain_occupation.csv  → taux occupation mensuel par hôtel")
print("  hotelchain_revpar.csv      → ADR, RevPAR mensuel par hôtel")
print("  hotelchain_clients_clv.csv → segmentation clients")
print("  hotelchain_canaux.csv      → performance par canal")
print(f"\n💡 Localisation : {SAVE_PATH}")

---
## Checklist de validation

| Étape | Contenu | Statut |
|---|---|---|
| 2 | 3 vues de nettoyage créées | [ ] |
| 3 | KPIs globaux calculés | [ ] |
| 4 | Taux d'occupation mensuel + visualisation | [ ] |
| 5 | Performance canaux avec RANK() et LAG() | [ ] |
| 6 | RevPAR calculé par hôtel et par mois | [ ] |
| 7 | Segmentation clients NTILE + VIP | [ ] |
| 8 | Heatmap saisonnalité produite | [ ] |
| 9 | Fonction rapport_hotel() testée | [ ] |
| 10 | 4 fichiers CSV exportés | [ ] |

| KPI | Valeur |
|---|---|
| Revenu total | ? FCFA |
| CSAT moyen | ? / 5 |
| Taux annulation | ? % |
| Hôtel meilleur RevPAR | ? |
| Canal le plus rentable | ? |

---

**DataProjectLab** — apprendre la data sur des cas concrets, structurés et orientés métier.